# 05 — Matching and Subclassification
**References:** Imbens & Rubin (2015) Ch. 15, 17–18 · Stuart (2010) "Matching Methods for Causal Inference" (*Statistical Science*) · Cochran (1968) · Morgan & Winship (2014) Ch. 5

## Narrative thread
```
Selection on observables -> Overlap -> Subclassification -> Exact & distance matching -> Bias-variance tradeoffs -> Balance diagnostics
```

## The identifying assumptions

Everything in notebooks 05–06 rests on two assumptions:

**1. Unconfoundedness (selection on observables, ignorability):**
$$(Y(0), Y(1)) \perp W \mid X$$
Within cells of $X$, treatment is as good as randomized. Untestable — justified only by
subject-matter knowledge (the backdoor criterion of notebook 03).

**2. Overlap (positivity):**
$$0 < P(W = 1 \mid X = x) < 1 \quad \text{for all } x \text{ in the support}$$
Every kind of unit must have a chance of being in both arms; otherwise the counterfactual
for some units is pure extrapolation.

Under both, the ATE is identified by conditioning-and-averaging (the g-formula):
$$\tau = E_X\big[\,E[Y \mid W=1, X] - E[Y \mid W=0, X]\,\big]$$

Matching and subclassification are **nonparametric** ways to compute this expectation —
they impute each unit's missing potential outcome from comparable units, instead of
trusting a global regression to extrapolate.

## Subclassification (Cochran 1968)

Split the sample into $K$ strata on a covariate (or on the propensity score, notebook 06),
take the within-stratum difference in means, and average with stratum weights:
$$\hat\tau = \sum_{k=1}^{K} \frac{n_k}{n} \left(\bar Y_{1k} - \bar Y_{0k}\right)$$
Cochran's classic result: ~5 strata remove roughly 90% of the bias from a single
smoothly-acting confounder.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [ ]:
# ── Cochran (1968) style example: smoking, age, and mortality ─────────────
# Treatment: cigarette smoking (vs pipe/cigar). Confounder: age.
# Pipe smokers are much older -> naive comparison flatters cigarettes.
n = 60_000
age = np.random.uniform(20, 80, n)
p_cig = 1 / (1 + np.exp((age - 45) / 8))          # younger people smoke cigarettes
w = np.random.binomial(1, p_cig)                   # 1 = cigarettes, 0 = pipe/cigar
# Mortality risk rises with age; cigarettes add a TRUE +5 points
mortality = 0.9*age + 5*w + np.random.normal(0, 6, n)

naive = mortality[w==1].mean() - mortality[w==0].mean()
print(f'True effect of cigarettes: +5.00')
print(f'Naive difference:          {naive:+.2f}   (cigarettes look protective!)')

# Subclassification on age with K strata
for K in [2, 5, 10, 20]:
    edges = np.quantile(age, np.linspace(0, 1, K + 1))
    stratum = np.clip(np.searchsorted(edges, age, side='right') - 1, 0, K - 1)
    tau_k, n_k = [], []
    for k in range(K):
        m = stratum == k
        if (w[m]==1).sum() > 0 and (w[m]==0).sum() > 0:
            tau_k.append(mortality[m][w[m]==1].mean() - mortality[m][w[m]==0].mean())
            n_k.append(m.sum())
    est = np.average(tau_k, weights=n_k)
    print(f'Subclassification K = {K:>2}:    {est:+.2f}')
print('\nA handful of strata removes most of the confounding bias (Cochran 1968).')

## Matching estimators

**Exact matching** pairs each treated unit with controls having identical $X$ — feasible
only for a few discrete covariates. With continuous $X$, match on a **distance**:

- **Mahalanobis distance:** $d(x_i, x_j) = \sqrt{(x_i - x_j)^\top \hat\Sigma^{-1} (x_i - x_j)}$ —
  scale-invariant, treats all covariates symmetrically.
- **Propensity-score distance** (notebook 06) — collapses $X$ to one dimension.

The 1-NN matching estimator of the **ATT**:
$$\hat\tau_{ATT} = \frac{1}{n_1} \sum_{i: W_i = 1} \left( Y_i - Y_{j(i)} \right)$$
where $j(i)$ is the nearest control to treated unit $i$.

Design choices and their bias-variance consequences:

| Choice | Bias | Variance |
|---|---|---|
| More matches per unit ($M \uparrow$) | ↑ (worse matches used) | ↓ |
| With replacement | ↓ (best match always available) | ↑ (controls reused) |
| Caliper (max distance) | ↓ | ↑ (unmatched treated dropped — estimand changes!) |

Matching discrepancies leave a residual bias of order $O(n^{-1/k})$ with $k$ continuous
covariates (Abadie & Imbens 2006) — the curse of dimensionality that motivates the
propensity score.


In [ ]:
# ── 1-NN Mahalanobis matching for the ATT, from scratch ──────────────────
from scipy.spatial.distance import cdist

def simulate_obs(n=3000, seed=0):
    rng = np.random.default_rng(seed)
    x1 = rng.normal(0, 1, n)                 # e.g. age (standardized)
    x2 = rng.normal(0, 1, n)                 # e.g. income (standardized)
    p  = 1 / (1 + np.exp(-(0.8*x1 + 0.8*x2)))
    w  = rng.binomial(1, p)
    y0 = 2*x1 + 2*x2 + rng.normal(0, 1, n)
    y1 = y0 + 3.0                            # constant effect: ATT = ATE = 3
    y  = np.where(w == 1, y1, y0)
    return pd.DataFrame(dict(x1=x1, x2=x2, w=w, y=y))

df = simulate_obs()
X = df[['x1', 'x2']].values
w, y = df['w'].values, df['y'].values

# Mahalanobis distance from every treated unit to every control
Sigma_inv = np.linalg.inv(np.cov(X.T))
D = cdist(X[w==1], X[w==0], metric='mahalanobis', VI=Sigma_inv)
match_idx = D.argmin(axis=1)                 # nearest control per treated (with replacement)

y_t = y[w==1]
y_c_matched = y[w==0][match_idx]
att_match = (y_t - y_c_matched).mean()

naive = y[w==1].mean() - y[w==0].mean()
ols   = smf.ols('y ~ w + x1 + x2', df).fit().params['w']
print(f'True ATT:                 3.000')
print(f'Naive difference:         {naive:.3f}')
print(f'OLS adjustment:           {ols:.3f}')
print(f'1-NN Mahalanobis matching: {att_match:.3f}')
print(f'\nControls reused: {len(match_idx) - len(np.unique(match_idx))} of {len(match_idx)} matches (with replacement)')

## Balance diagnostics: did the design work?

Matching is **design, not analysis**: assess covariate balance *without looking at
outcomes* (Rubin: "the design phase should be outcome-free"). The standard metric is the
**standardized mean difference**

$$\text{SMD}_j = \frac{\bar X_{j,1} - \bar X_{j,0}}{\sqrt{(s_{j,1}^2 + s_{j,0}^2)/2}}$$

with the conventional threshold $|\text{SMD}| < 0.1$ after matching. A **Love plot**
displays SMDs before vs. after. Note: balance tests with p-values are discouraged —
balance is a property of the *sample*, not a hypothesis about a population (Imai, King &
Stuart 2008).


In [ ]:
# ── Love plot: standardized mean differences before/after matching ───────
def smd(a, b):
    return (a.mean() - b.mean()) / np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)

covs = ['x1', 'x2']
Xt, Xc = X[w==1], X[w==0]
Xc_matched = X[w==0][match_idx]

smd_before = [smd(Xt[:, j], Xc[:, j]) for j in range(2)]
smd_after  = [smd(Xt[:, j], Xc_matched[:, j]) for j in range(2)]

fig, ax = plt.subplots(figsize=(7, 2.8))
ypos = np.arange(len(covs))
ax.scatter(np.abs(smd_before), ypos, s=60, label='before matching', color='#d62728')
ax.scatter(np.abs(smd_after), ypos, s=60, label='after matching', color='#2ca02c')
ax.axvline(0.1, color='k', ls='--', lw=1, label='0.1 threshold')
ax.set_yticks(ypos, covs); ax.set_xlabel('|standardized mean difference|')
ax.set_title('Love plot: covariate balance'); ax.legend()
plt.tight_layout(); plt.show()

print(f'{"covariate":<10} {"SMD before":>11} {"SMD after":>10}')
for c, b, a in zip(covs, smd_before, smd_after):
    print(f'{c:<10} {b:11.3f} {a:10.3f}')

## Key takeaways

| Concept | Statement |
|---|---|
| Unconfoundedness | $(Y(0),Y(1)) \perp W \mid X$ — untestable, justified by the DAG |
| Overlap | $0 < e(x) < 1$; without it, estimation = extrapolation |
| Subclassification | ~5 strata kill ~90% of one confounder's bias (Cochran) |
| 1-NN matching | Impute the counterfactual from the nearest comparable unit |
| With replacement / M / caliper | Bias-variance dials; calipers change the estimand |
| Curse of dimensionality | Matching bias grows with the number of continuous covariates |
| Balance first | Judge the design by SMDs (< 0.1), never by outcome models |
